### Imports and Configuration

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Catalog
CATALOG = "merchai_data"
SCHEMA = "bronze"

RAW_PATH = "/Volumes/merchai_data/default/raw_data"


### Schemas

In [0]:
orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("user_id", IntegerType(), False),
    StructField("eval_set", StringType(), True),
    StructField("order_number", IntegerType(), True),
    StructField("order_dow", IntegerType(), True),
    StructField("order_hour_of_day", IntegerType(), True),
    StructField("days_since_prior_order", FloatType(), True)
])

In [0]:
products_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("aisle_id", IntegerType(), True),
    StructField("department_id", IntegerType(), True)
])

In [0]:
departments_schema = StructType([
    StructField("department_id", IntegerType(), False),
    StructField("department", StringType(), True)
])

In [0]:
aisles_schema = StructType([
    StructField("aisle_id", IntegerType(), False),
    StructField("aisle", StringType(), True)
])

In [0]:
order_products_prior_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("add_to_cart_order", IntegerType(), True),
    StructField("reordered", IntegerType(), True)
])

In [0]:
order_products_train_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("add_to_cart_order", IntegerType(), True),
    StructField("reordered", IntegerType(), True)
])

## Reading CSV Files

In [0]:
orders_df = spark.read \
    .option("header", True) \
    .schema(orders_schema) \
    .csv(f"{RAW_PATH}/orders.csv")

products_df = spark.read \
    .option("header", True) \
    .schema(products_schema) \
    .csv(f"{RAW_PATH}/products.csv")

departments_df = spark.read \
    .option("header", True) \
    .schema(departments_schema) \
    .csv(f"{RAW_PATH}/departments.csv")

aisles_df = spark.read \
    .option("header", True) \
    .schema(aisles_schema) \
    .csv(f"{RAW_PATH}/aisles.csv")

order_products_prior_df = spark.read \
    .option("header", True) \
    .schema(order_products_prior_schema) \
    .csv(f"/Volumes/merchai_data/default/raw_data/order_products__prior.csv")
    

order_products_train_df = spark.read \
    .option("header", True) \
    .schema(order_products_train_schema) \
    .csv(f"/Volumes/merchai_data/default/raw_data/order_products__train.csv")

## Displaying Data

In [0]:
print("Orders")
display(orders_df.limit(10))

print("Products")
display(products_df.limit(10))

print("Departments")
display(departments_df.limit(10))

print("Aisles")
display(aisles_df.limit(10))

print("Order Products Prior")
display(order_products_prior_df.limit(10))

print("Order Products Train")
display(order_products_train_df.limit(10))

## Records Counting

In [0]:
print("Orders:", orders_df.count())
print("Products:", products_df.count())
print("Departments:", departments_df.count())
print("Aisles:", aisles_df.count())
print("Prior:", order_products_prior_df.count())
print("Train:", order_products_train_df.count())

## Writing Bronze Delta Table

In [0]:
orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.orders")

products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.products")

departments_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.departments")

aisles_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.aisles")

order_products_prior_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.order_products_prior")

order_products_train_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.order_products_train")

In [0]:
%sql
SHOW TABLES IN merchai_data.bronze;